In [1]:
from pythtb import Lattice, Mesh, TBModel, WFArray
import numpy as np

In [2]:
lattice = Lattice(
    lat_vecs=[[1.0]], 
    orb_vecs=[[0.0], [1 / 3], [2 / 3]], 
    periodic_dirs=[0]
    )

In [3]:
mesh = Mesh(dim_k=1, axis_types=["k"])

In [4]:
mesh.build_grid([20])

In [5]:
wfa = WFArray(lattice=lattice, mesh=mesh)

In [6]:
def set_model(t: float, delta: float, lmbda: float) -> TBModel:
    """Periodic three-site model at parameter lmbda."""
   
    model = TBModel(lattice=lattice)

    # nearest-neighbour hoppings (last hop wraps to the next cell)
    model.set_hop(t, 0, 1, [0])
    model.set_hop(t, 1, 2, [0])
    model.set_hop(t, 2, 0, [1])

    onsite = [delta * -np.cos(2 * np.pi * (lmbda - idx / 3)) for idx in range(3)]
    model.set_onsite(onsite)
    return model

model = set_model(t=1.0, delta=1.0, lmbda=0.0)
wfa.solve_model(model)

In [7]:
wfa[2]

array([[ 0.87598308-0.j        , -0.2806304 +0.1938386j ,
        -0.2806304 -0.1938386j ],
       [ 0.26138584+0.j        , -0.06155942-0.67974198j,
        -0.06155942+0.67974198j],
       [ 0.40537771+0.j        ,  0.64610914+0.0194278j ,
         0.64610914-0.0194278j ]])

In [8]:
mesh = Mesh(
    dim_k=1,
    dim_lambda=1,
    axis_types=["k", "l"],   # first axis: crystal momentum; second: adiabatic parameter
    axis_names=["kx", "lmbda"],
)

In [9]:
mesh.build_grid(shape=(31, 11), gamma_centered=True, lambda_start=0.0, lambda_stop=1.0)

In [10]:
mesh.loop_axis(axis_idx=1, component_idx=1)  # form the lambda axis into a loop

In [11]:
mesh.close_axis(axis_idx=1, component_idx=1) # indicate that the end of the loop completes the cycle (endpoint included)
print(mesh)

Mesh Summary
Type: grid
Dimensionality: 1 k-dim(s) + 1 λ-dim(s)
Number of mesh points: 341
Full shape: (31, 11, 2)
k-axes: [Axis(type=k, name=kx, size=31)]
λ-axes: [Axis(type=l, name=lmbda, size=11)]
Is a torus in k-space (all k-axes wind BZ): yes
Looped axes: (axis 0 loops component 0), (axis 1 loops component 1)
BZ-winding axes: (axis 0 winds component 0)
Endpoint axes: (axis 1 contains endpoint of component 1)


In [12]:
wfa = WFArray(lattice, mesh)

In [13]:
t = -1.3
delta = 2.0
fixed_params = {"t": t, "delta": delta}

In [14]:
wfa.solve_model(model_func=set_model, fixed_params=fixed_params)